# Notebook note

The saved width-sweep results are used by the figure script. Use this notebook when rerunning the sweep or changing the widths/seeds.


# ReLU width sweep

This sweep keeps $\bar h=0.1$ fixed and changes the hidden width. It checks where the mean-field picture starts to look reliable, and where finite-width effects still dominate the scheduling signal.

Schedules: no dropout, constant dropout, Step (early, $2\bar h$), and Big step (early, $3\bar h$).


In [ ]:
from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "dropout_mft").exists():
        sys.path.insert(0, str(_root / "src"))
        break

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm
import contextlib
import warnings
import pickle
import ast
import time

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = (device == 'cuda')
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Settings

N_SIMULATIONS = 10

# Fixed mean dropout field
H_BAR = 0.1

# Width sweep: 64, 128, 256, 512, 1024
WIDTH_VALUES = [64, 128, 256, 512, 1024]

# Schedules to compare
SCHEDULES = ['none', 'constant', 'reverse_step', 'big_step']

# Network geometry (depth is fixed across all widths)
SIGMA_B_SQ = 0.00
SIGMA_W_SQ = 2
DEPTH_MULTIPLIER = 2

# Depth is fixed using the h_bar = 0.1 correlation length.
kappa = 2 * np.sqrt(2) / (3 * np.pi)
alpha = (3 / 2) * kappa ** (2 / 3)
xi_ref = 1 / (alpha * H_BAR ** (1 / 3))
DEPTH = int(DEPTH_MULTIPLIER * xi_ref)

# Training
EPOCHS = 75
LEARNING_RATE = 1e-4
BATCH_SIZE = 75
USE_LR_SCHEDULE = True
LR_MIN = 1e-7

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000

# Plotting - selected width for single-width training curves figure
WIDTH_PLOT = 256  # must be in WIDTH_VALUES

from dropout_mft.schedules import DISPLAY_NAMES, MARKERS
from dropout_mft.style import SCHEDULE_COLORS as COLORS

print(f"Fixed h_bar: {H_BAR}")
print(f"Fixed depth: L = {DEPTH}  (from {DEPTH_MULTIPLIER} x xi at h_bar = {H_BAR})")
print(f"Width values: {WIDTH_VALUES}")
print(f"L/N ratios:   {[round(DEPTH/w, 3) for w in WIDTH_VALUES]}")
print(f"Schedules: {SCHEDULES}")
print(f"Simulations per config: {N_SIMULATIONS}")
total_runs = len(WIDTH_VALUES) * len(SCHEDULES)
print(f"Total training runs: {total_runs * N_SIMULATIONS}")


In [ ]:
# Dropout schedules

def get_dropout_schedule(schedule_type, depth, h_bar, h_max=None):
    """
    Generate layer-wise dropout rates.
    All schedules have mean dropout = h_bar.
    """
    if schedule_type == 'none':
        return [0.0] * depth

    if schedule_type == 'constant':
        return [h_bar] * depth

    if schedule_type == 'reverse_step':
        # Dropout concentrated in first half (early), mean = h_bar
        if h_max is None:
            h_max = 2 * h_bar
        f = h_bar / h_max
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [h_adj] * n_drop + [0.0] * (depth - n_drop)

    if schedule_type == 'big_step':
        # Dropout concentrated in first 1/3 of layers, mean = h_bar
        f = 1 / 3
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [h_adj] * n_drop + [0.0] * (depth - n_drop)

    raise ValueError(f"Unknown schedule: {schedule_type}")


def compute_effective_xi(h_layers):
    """Compute effective correlation length for ReLU (kinked) networks."""
    L = len(h_layers)
    damage = sum(h ** (1 / 3) for h in h_layers if h > 0) / L
    xi_inv = alpha * damage
    return 1 / xi_inv if xi_inv > 0 else float('inf')

In [ ]:
# Schedule diagnostics for each width

theory = {}

print(f"{'Width':<8} {'Schedule':<16} {'h_mean':>8} {'h_max':>8} {'xi_eff':>8} {'L/xi':>6} {'L/N':>6}")
print('-' * 70)

h_max = 2 * H_BAR
for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, h_max)
        xi = compute_effective_xi(h_layers)
        theory[(width, sched)] = {'h_layers': h_layers, 'xi_eff': xi}

        xi_str = f'{xi:.2f}' if xi < 1e6 else 'inf'
        ratio = DEPTH / xi if xi < 1e6 else 0
        print(f'{width:<8d} {sched:<16} {np.mean(h_layers):>8.4f} '
              f'{max(h_layers):>8.4f} {xi_str:>8} {ratio:>6.2f} {DEPTH/width:>6.3f}')


In [ ]:
# Model

class CriticalReLUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, h_layers,
                 sigma_w_sq=2.0, sigma_b_sq=0.0):
        super().__init__()
        self.depth = len(h_layers)
        self.sigma_w_sq = sigma_w_sq
        self.sigma_b_sq = sigma_b_sq

        layers = [nn.Linear(input_dim, hidden_dim)]
        for _ in range(self.depth - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
        layers.append(nn.Linear(hidden_dim, output_dim))

        self.layers = nn.ModuleList(layers)
        self.dropouts = nn.ModuleList([nn.Dropout(p=h) for h in h_layers])
        self._init_critical()

    def _init_critical(self):
        for layer in self.layers:
            fan_in = layer.weight.shape[1]
            std_w = np.sqrt(self.sigma_w_sq / fan_in)
            nn.init.normal_(layer.weight, mean=0.0, std=std_w)
            if self.sigma_b_sq > 0:
                nn.init.normal_(layer.bias, mean=0.0, std=np.sqrt(self.sigma_b_sq))
            else:
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = torch.relu(layer(x))
            x = self.dropouts[i](x)
        return self.layers[-1](x)

In [ ]:
# Training code

def autocast_context():
    if not USE_AMP:
        return contextlib.nullcontext()
    return autocast(enabled=True, dtype=AMP_DTYPE)


def iterate_batches(x, y, batch_size, shuffle=True):
    n = x.shape[0]
    idx = torch.randperm(n, device=x.device) if shuffle else torch.arange(n, device=x.device)
    for i in range(0, n, batch_size):
        yield x[idx[i:i + batch_size]], y[idx[i:i + batch_size]]


def load_cifar10(train_size=None, test_size=None, seed=0):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

    train_set = datasets.CIFAR10('./data', train=True, download=True)
    test_set = datasets.CIFAR10('./data', train=False, download=True)

    rng = np.random.RandomState(seed)

    if train_size is None or train_size >= len(train_set.data):
        tr_idx = np.arange(len(train_set.data))
    else:
        tr_idx = rng.choice(len(train_set.data), train_size, replace=False)

    if test_size is None or test_size >= len(test_set.data):
        te_idx = np.arange(len(test_set.data))
    else:
        te_idx = rng.choice(len(test_set.data), test_size, replace=False)

    print(f"Train: {len(tr_idx)} samples, Test: {len(te_idx)} samples")

    x_tr = torch.from_numpy(train_set.data[tr_idx]).permute(0, 3, 1, 2).float() / 255
    y_tr = torch.tensor(np.array(train_set.targets)[tr_idx])
    x_te = torch.from_numpy(test_set.data[te_idx]).permute(0, 3, 1, 2).float() / 255
    y_te = torch.tensor(np.array(test_set.targets)[te_idx])

    if device == 'cuda':
        x_tr, y_tr = x_tr.cuda(), y_tr.cuda()
        x_te, y_te = x_te.cuda(), y_te.cuda()
        mean, std = mean.cuda(), std.cuda()

    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    return (x_tr, y_tr), (x_te, y_te)


def train_epoch(model, data, optimizer, criterion, batch_size):
    model.train()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in iterate_batches(x, y, batch_size):
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            out = model(xb)
            loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum / total, 100 * correct / total


@torch.no_grad()
def evaluate(model, data, criterion, batch_size):
    model.eval()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in iterate_batches(x, y, batch_size, shuffle=False):
        with autocast_context():
            out = model(xb)
            loss = criterion(out, yb)
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum / total, 100 * correct / total

In [ ]:
# Data

print("Loading CIFAR-10...")
train_data, test_data = load_cifar10(TRAIN_SIZE, TEST_SIZE)
print(f"Train shape: {train_data[0].shape}")

In [ ]:
# Run the width sweep

def run_simulations(h_layers, width, n_sims, label):
    """Train N_SIMULATIONS independent runs and return aggregated result dict."""
    histories = []
    for sim in range(n_sims):
        seed = 42 + sim
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = CriticalReLUNet(
            input_dim=3072, hidden_dim=width, output_dim=10,
            h_layers=h_layers,
            sigma_w_sq=SIGMA_W_SQ, sigma_b_sq=SIGMA_B_SQ
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-7)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN) if USE_LR_SCHEDULE else None
        criterion = nn.CrossEntropyLoss()

        hist = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
        best_test_acc = 0.0
        pbar = tqdm(range(EPOCHS), desc=f'{label} sim {sim+1}/{n_sims}', leave=True)

        for ep in pbar:
            tr_loss, tr_acc = train_epoch(model, train_data, optimizer, criterion, BATCH_SIZE)
            if scheduler:
                scheduler.step()
            te_loss, te_acc = evaluate(model, test_data, criterion, BATCH_SIZE)
            best_test_acc = max(best_test_acc, te_acc)
            hist['train_acc'].append(tr_acc)
            hist['test_acc'].append(te_acc)
            hist['train_loss'].append(tr_loss)
            hist['test_loss'].append(te_loss)
            pbar.set_postfix({'tr_L': f'{tr_loss:.3f}', 'te_A': f'{te_acc:.1f}%', 'best': f'{best_test_acc:.1f}%'})

        histories.append(hist)

    return {
        'train_acc':  np.array([h['train_acc']  for h in histories]),
        'test_acc':   np.array([h['test_acc']   for h in histories]),
        'train_loss': np.array([h['train_loss'] for h in histories]),
        'test_loss':  np.array([h['test_loss']  for h in histories]),
    }


all_results = {}
t_start = time.time()

h_max = 2 * H_BAR

# Architecture changes with width, so every schedule (including 'none') is
# run separately for each width.
for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        print(f"\n{'='*60}")
        print(f"Width = {width} | Schedule: {sched}")
        print(f"{'='*60}")

        h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, h_max)
        xi = compute_effective_xi(h_layers)
        if sched == 'none':
            print(f"  xi_eff = inf (no dropout), L/N = {DEPTH/width:.3f}")
        else:
            print(f"  h_layers = {[f'{h:.4f}' for h in h_layers]}")
            print(f"  xi_eff = {xi:.2f}, L/xi = {DEPTH/xi:.2f}, L/N = {DEPTH/width:.3f}")

        result = run_simulations(h_layers, width, N_SIMULATIONS, f'w={width} {sched}')
        all_results[(width, sched)] = result

        best = result['test_acc'].max(axis=1)
        print(f"  Best test = {best.mean():.2f}% +/- {best.std()/np.sqrt(len(best)):.2f}% (SEM)")

elapsed = time.time() - t_start
print(f"\nTotal time: {elapsed/60:.1f} min")


In [ ]:
# Save

save_data = {
    'all_results': {str(k): v for k, v in all_results.items()},
    'theory': {str(k): v for k, v in theory.items()},
    'config': {
        'H_BAR': H_BAR,
        'WIDTH_VALUES': WIDTH_VALUES,
        'SCHEDULES': SCHEDULES,
        'N_SIMULATIONS': N_SIMULATIONS,
        'EPOCHS': EPOCHS,
        'DEPTH': DEPTH,
        'SIGMA_W_SQ': SIGMA_W_SQ,
        'SIGMA_B_SQ': SIGMA_B_SQ,
        'LEARNING_RATE': LEARNING_RATE,
    }
}

with open('width_sweep_results.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("Results saved to width_sweep_results.pkl")


In [ ]:
# Load a saved sweep

with open('width_sweep_results.pkl', 'rb') as f:
    save_data = pickle.load(f)

# Convert stringified tuple keys back to tuples
all_results = {ast.literal_eval(k): v for k, v in save_data['all_results'].items()}
theory      = {ast.literal_eval(k): v for k, v in save_data['theory'].items()}
config      = save_data['config']

# Put config names back where the plotting cells expect them
H_BAR         = config['H_BAR']
WIDTH_VALUES  = config['WIDTH_VALUES']
SCHEDULES     = config['SCHEDULES']
N_SIMULATIONS = config['N_SIMULATIONS']
EPOCHS        = config['EPOCHS']
DEPTH         = config['DEPTH']
SIGMA_W_SQ    = config['SIGMA_W_SQ']
SIGMA_B_SQ    = config['SIGMA_B_SQ']
LEARNING_RATE = config['LEARNING_RATE']

print(f"Loaded {len(all_results)} result entries from width_sweep_results.pkl")
print(f"h_bar: {H_BAR}")
print(f"Width values: {WIDTH_VALUES}")
print(f"Schedules: {SCHEDULES}")


In [ ]:
# Summary table

print(f"{'Width':<8} {'Schedule':<16} {'xi_eff':>8} {'L/xi':>6} {'L/N':>6} {'Best Test Acc':>20}")
print('=' * 75)

for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        xi = theory[(width, sched)]['xi_eff']
        best = all_results[(width, sched)]['test_acc'].max(axis=1)
        sem = best.std() / np.sqrt(len(best))
        xi_str = f'{xi:.2f}' if xi < 1e6 else 'inf'
        ratio = f'{DEPTH/xi:.2f}' if xi < 1e6 else '0.00'
        print(f'{width:<8d} {DISPLAY_NAMES[sched]:<16} {xi_str:>8} {ratio:>6} '
              f'{DEPTH/width:>6.3f} {best.mean():>8.2f} +/- {sem:.2f}%')
    print('-' * 75)


In [ ]:
# Plot style

plt.style.use('seaborn-v0_8-whitegrid')

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'lines.linewidth': 1.5,
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'mathtext.fontset': 'cm',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
})

In [ ]:
# Best test accuracy versus width

fig, ax = plt.subplots(figsize=(6, 4))

for sched in SCHEDULES:
    means, sems = [], []
    for width in WIDTH_VALUES:
        best = all_results[(width, sched)]['test_acc'].max(axis=1)
        means.append(best.mean())
        sems.append(best.std() / np.sqrt(len(best)))

    ax.errorbar(WIDTH_VALUES, means, yerr=sems,
                fmt=MARKERS[sched] + '-', color=COLORS[sched],
                label=DISPLAY_NAMES[sched], capsize=3, capthick=1.2,
                markersize=6, markeredgecolor='white', markeredgewidth=0.5)

ax.set_xscale('log', base=2)
ax.set_xticks(WIDTH_VALUES)
ax.set_xticklabels([str(w) for w in WIDTH_VALUES])
ax.set_xlabel(r'Hidden width $N$')
ax.set_ylabel('Best test accuracy (%)')
ax.set_title(rf'Dropout scheduling vs width at $\bar{{h}}={H_BAR}$ (MLP, CIFAR-10)')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig('width_sweep_accuracy.pdf')
plt.show()
print("Saved: width_sweep_accuracy.pdf")


In [ ]:
# Scheduling advantage over constant dropout

fig, ax = plt.subplots(figsize=(6, 3.5))

deltas_step, deltas_step_sem = [], []
deltas_bigstep, deltas_bigstep_sem = [], []

for width in WIDTH_VALUES:
    best_const = all_results[(width, 'constant')]['test_acc'].max(axis=1)
    best_step = all_results[(width, 'reverse_step')]['test_acc'].max(axis=1)
    best_bigstep = all_results[(width, 'big_step')]['test_acc'].max(axis=1)

    d_step = best_step - best_const
    deltas_step.append(d_step.mean())
    deltas_step_sem.append(d_step.std() / np.sqrt(len(d_step)))

    d_bigstep = best_bigstep - best_const
    deltas_bigstep.append(d_bigstep.mean())
    deltas_bigstep_sem.append(d_bigstep.std() / np.sqrt(len(d_bigstep)))

ax.errorbar(WIDTH_VALUES, deltas_step, yerr=deltas_step_sem,
            fmt='D-', color=COLORS['reverse_step'],
            label='Step (early) - Constant', capsize=3, markersize=6,
            markeredgecolor='white', markeredgewidth=0.5)

ax.errorbar(WIDTH_VALUES, deltas_bigstep, yerr=deltas_bigstep_sem,
            fmt='v-', color=COLORS['big_step'],
            label='Big step (1/3) - Constant', capsize=3, markersize=6,
            markeredgecolor='white', markeredgewidth=0.5)

ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xscale('log', base=2)
ax.set_xticks(WIDTH_VALUES)
ax.set_xticklabels([str(w) for w in WIDTH_VALUES])
ax.set_xlabel(r'Hidden width $N$')
ax.set_ylabel(r'$\Delta$ best test accuracy (%)')
ax.set_title(rf'Scheduling advantage vs width at $\bar{{h}}={H_BAR}$')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig('width_sweep_delta.pdf')
plt.show()
print("Saved: width_sweep_delta.pdf")


In [ ]:
# Accuracy versus L/N

fig, ax = plt.subplots(figsize=(6, 4))

ratios = [DEPTH / w for w in WIDTH_VALUES]

for sched in SCHEDULES:
    means, sems = [], []
    for width in WIDTH_VALUES:
        best = all_results[(width, sched)]['test_acc'].max(axis=1)
        means.append(best.mean())
        sems.append(best.std() / np.sqrt(len(best)))

    ax.errorbar(ratios, means, yerr=sems,
                fmt=MARKERS[sched] + '-', color=COLORS[sched],
                label=DISPLAY_NAMES[sched], capsize=3, markersize=6,
                markeredgecolor='white', markeredgewidth=0.5)

ax.set_xscale('log')
ax.set_xlabel(r'Aspect ratio $L/N$')
ax.set_ylabel('Best test accuracy (%)')
ax.set_title(rf'Accuracy vs aspect ratio at $\bar{{h}}={H_BAR}$')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig('width_sweep_aspect_ratio.pdf')
plt.show()
print("Saved: width_sweep_aspect_ratio.pdf")


In [ ]:
# =============================================================================
# Table values used in the paper
# =============================================================================

print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Best test accuracy (\%) across network widths $N$ at fixed $\bar{h}=", end='')
print(f"{H_BAR}", end='')
print(r"$ for different schedules (MLP, CIFAR-10). Errors are SEM over ", end='')
print(f"{N_SIMULATIONS}", end='')
print(r" seeds.}")
print(r"\label{tab:width_sweep}")
print(r"\begin{tabular}{c" + "c" * len(SCHEDULES) + r"}")
print(r"\toprule")

header = r"$N$"
for sched in SCHEDULES:
    header += f" & {DISPLAY_NAMES[sched]}"
header += r" \\"
print(header)
print(r"\midrule")

for width in WIDTH_VALUES:
    row = f"{width}"
    vals = []
    for sched in SCHEDULES:
        best = all_results[(width, sched)]['test_acc'].max(axis=1)
        sem = best.std() / np.sqrt(len(best))
        vals.append((best.mean(), sem))

    # Best dropout schedule, ignoring the no-dropout row
    dropout_vals = [(i, v[0]) for i, v in enumerate(vals) if SCHEDULES[i] != 'none']
    best_idx = max(dropout_vals, key=lambda x: x[1])[0] if dropout_vals else -1

    for i, (mean, sem) in enumerate(vals):
        if i == best_idx:
            row += f" & \\textbf{{{mean:.2f}}} $\\pm$ {sem:.2f}"
        else:
            row += f" & {mean:.2f} $\\pm$ {sem:.2f}"

    row += r" \\"
    print(row)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


In [ ]:
# Training curves for one width

fig, axes = plt.subplots(2, 2, figsize=(8, 6))
lines, labels = [], []
ep = np.arange(EPOCHS)

for sched in SCHEDULES:
    key = (WIDTH_PLOT, sched)
    if key not in all_results:
        continue
    res = all_results[key]
    c = COLORS[sched]
    label = DISPLAY_NAMES[sched]

    # Training loss
    m = res['train_loss'].mean(0)
    s = res['train_loss'].std(0)
    axes[0,0].semilogy(ep, m, color=c, lw=1.5)
    axes[0,0].fill_between(ep, np.maximum(m-s, 1e-6), m+s, color=c, alpha=0.08)

    # Test loss
    m = res['test_loss'].mean(0)
    s = res['test_loss'].std(0)
    axes[0,1].semilogy(ep, m, color=c, lw=1.5)
    axes[0,1].fill_between(ep, np.maximum(m-s, 1e-6), m+s, color=c, alpha=0.08)

    # Train accuracy
    m = res['train_acc'].mean(0)
    s = res['train_acc'].std(0)
    axes[1,0].plot(ep, m, color=c, lw=1.5)
    axes[1,0].fill_between(ep, m-s, m+s, color=c, alpha=0.08)

    # Test accuracy
    m = res['test_acc'].mean(0)
    s = res['test_acc'].std(0)
    line, = axes[1,1].plot(ep, m, color=c, lw=1.5)
    axes[1,1].fill_between(ep, m-s, m+s, color=c, alpha=0.08)
    lines.append(line)
    labels.append(label)

axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Training loss')
axes[0,0].set_title('(a) Training loss', loc='left')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Test loss')
axes[0,1].set_title('(b) Test loss', loc='left')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Training accuracy (%)')
axes[1,0].set_title('(c) Training accuracy', loc='left')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Test accuracy (%)')
axes[1,1].set_title('(d) Test accuracy', loc='left')

fig.suptitle(rf'Training curves at $N = {WIDTH_PLOT}$, $\bar{{h}}={H_BAR}$', fontsize=13, y=1.02)
fig.legend(lines, labels, loc='lower center', ncol=len(labels), fontsize=8,
           framealpha=0.95, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(f'width_{WIDTH_PLOT}_training_curves.pdf', bbox_inches='tight')
plt.savefig(f'width_{WIDTH_PLOT}_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Training curves across widths

n_w = len(WIDTH_VALUES)
fig, axes = plt.subplots(n_w, 2, figsize=(10, 2.4 * n_w), sharex=True)
ep = np.arange(EPOCHS)
lines, labels = [], []

for row, width in enumerate(WIDTH_VALUES):
    for sched in SCHEDULES:
        key = (width, sched)
        if key not in all_results:
            continue
        res = all_results[key]
        c = COLORS[sched]

        # Test loss (left column)
        m = res['test_loss'].mean(0)
        s = res['test_loss'].std(0)
        axes[row, 0].semilogy(ep, m, color=c, lw=1.3)
        axes[row, 0].fill_between(ep, np.maximum(m - s, 1e-6), m + s, color=c, alpha=0.08)

        # Test accuracy (right column)
        m = res['test_acc'].mean(0)
        s = res['test_acc'].std(0)
        line, = axes[row, 1].plot(ep, m, color=c, lw=1.3)
        axes[row, 1].fill_between(ep, m - s, m + s, color=c, alpha=0.08)

        if row == 0:
            lines.append(line)
            labels.append(DISPLAY_NAMES[sched])

    # Row label
    axes[row, 0].set_ylabel('Test loss')
    axes[row, 1].set_ylabel('Test acc (%)')
    for col in range(2):
        axes[row, col].annotate(
            rf'$N={width}$', xy=(0.97, 0.95), xycoords='axes fraction',
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.8', alpha=0.9))

# Bottom x-labels only
axes[-1, 0].set_xlabel('Epoch')
axes[-1, 1].set_xlabel('Epoch')

# Column titles
axes[0, 0].set_title('Test loss', fontsize=11)
axes[0, 1].set_title('Test accuracy', fontsize=11)

fig.legend(lines, labels, loc='lower center', ncol=len(labels), fontsize=9,
           framealpha=0.95, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('width_sweep_all_curves.pdf', bbox_inches='tight')
plt.savefig('width_sweep_all_curves.png', dpi=300, bbox_inches='tight')
plt.show()
